# 06 — Reactive Changelog

A changelog that rebuilds automatically when data changes.

**What you learn:**
- `ReactiveManager` with `subscribe=True` for automatic updates
- `data_formula` to compute grouped sections from raw entries
- Changing data triggers formula recalculation automatically
- The document reflects the new state without manual rebuild

**Prerequisites:** 05_github_pr

In [ ]:
from IPython.display import Markdown
from genro_builders.contrib.markdown import MarkdownBuilder
from genro_builders.manager import ReactiveManager


class Changelog(ReactiveManager):
    """Reactive changelog — add entries, document updates itself."""

    def __init__(self):
        self.doc = self.set_builder('doc', MarkdownBuilder)
        self.run(subscribe=True)

    def store(self, data):
        data['version'] = '1.2.0'
        data['date'] = '2026-04-13'
        data['entries'] = [
            ('feat', 'Add SSO callback validation'),
            ('feat', 'Add dark mode support'),
            ('fix', 'Fix CORS headers on API responses'),
            ('fix', 'Fix pagination offset in /items endpoint'),
            ('docs', 'Update API reference for v1.2'),
        ]

    def main(self, source):
        source.data_setter('version', value='^version')
        source.data_setter('date', value='^date')
        source.data_setter('entries', value='^entries')

        # Computed: group entries by type
        def group_entries(entries):
            groups = {}
            for entry_type, description in entries:
                groups.setdefault(entry_type, []).append(description)
            return groups

        source.data_formula(
            'grouped',
            func=group_entries,
            entries='^entries',
        )

        # Computed: formatted changelog text
        def format_changelog(grouped):
            type_labels = {
                'feat': 'Features',
                'fix': 'Bug Fixes',
                'docs': 'Documentation',
                'refactor': 'Refactoring',
                'test': 'Tests',
            }
            sections = []
            for entry_type, items in grouped.items():
                label = type_labels.get(entry_type, entry_type.capitalize())
                lines = [f'### {label}']
                for item in items:
                    lines.append(f'- {item}')
                sections.append('\n'.join(lines))
            return '\n\n'.join(sections)

        source.data_formula(
            'changelog_body',
            func=format_changelog,
            grouped='^grouped',
        )

        # Computed: entry count
        source.data_formula(
            'entry_count',
            func=lambda entries: len(entries),
            entries='^entries',
        )

        # --- Structure ---
        source.h1('^version')
        source.p('^date')
        source.p('^changelog_body')

## Initial changelog

In [ ]:
app = Changelog()
store = app.reactive_store

print(f"Entries: {store['entry_count']}")
print(f"Groups: {list(store['grouped'].keys())}")
print()
Markdown(app.doc.render())

## Add entries — formulas recalculate automatically

With `ReactiveManager` and `subscribe=True`, changing the data
triggers formula recalculation. The grouped sections and count
update without calling `build()` manually.

In [ ]:
# Add new entries — including a new type 'refactor'
current = list(store['entries'])
current.append(('feat', 'Add webhook retry with exponential backoff'))
current.append(('refactor', 'Extract auth middleware into dedicated module'))
current.append(('fix', 'Fix memory leak in WebSocket handler'))
store['entries'] = current

print(f"Entries: {store['entry_count']}")
print(f"Groups: {list(store['grouped'].keys())}")

In [ ]:
# Rebuild to reflect new data in the rendered output
app.doc.build()
Markdown(app.doc.render())

## Bump version

Change version and date — the header updates:

In [ ]:
store['version'] = '1.3.0'
store['date'] = '2026-05-01'

app.doc.build()
Markdown(app.doc.render())